# Fraud Detection Pipeline (`orders.is_fraud`) using CRISP-DM

## CRISP-DM Roadmap
1. **Business Understanding** *(this notebook section now)*
2. **Data Understanding** *(this notebook section now)*
3. **Data Preparation** *(next: feature engineering + train/validation split strategy)*
4. **Modeling** *(next: baseline + tuned models)*
5. **Evaluation** *(next: thresholding + business-cost analysis)*
6. **Deployment** *(next: batch/real-time scoring + monitoring)*

This notebook currently implements the first two phases in depth and prepares outputs needed for phases 3-6.

## 1) Business Understanding

### Fraud-detection problem definition
We need to predict whether an order in the `orders` table is fraudulent (`is_fraud = 1`) before fulfillment decisions are finalized. This is a **binary classification** problem with class imbalance.

### Business objective
Reduce fraud losses while minimizing friction for legitimate customers.

### Decision context
Model outputs will be used to support operational actions such as:
- auto-approve low-risk orders,
- route medium-risk orders to manual review,
- auto-hold/cancel very high-risk orders.

### Success criteria
**Business KPIs**
- Lower net fraud loss rate (chargebacks + write-offs).
- Keep false-positive rate low enough to avoid unnecessary customer friction.
- Maintain acceptable manual-review volume.

**ML metrics**
- Prioritize **Recall** and **PR-AUC** for fraud class.
- Track precision at an operating threshold (e.g., top-k risk queue).
- Track ROC-AUC as a secondary separability metric.

### Constraints and assumptions
- Data source is `shop.db`; target leakage must be avoided.
- Features available at scoring time should be preferred.
- Since fraud prevalence is low, threshold selection must be business-cost aware.

In [1]:
# 2) Data Understanding - setup and data loading
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DB_PATH = "shop.db"

conn = sqlite3.connect(DB_PATH)

# Load core tables needed for understanding fraud dynamics
orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
shipments_df = pd.read_sql_query("SELECT * FROM shipments", conn)
order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)

print("orders shape:", orders_df.shape)
print("customers shape:", customers_df.shape)
print("shipments shape:", shipments_df.shape)
print("order_items shape:", order_items_df.shape)
orders_df.head()

orders shape: (5000, 17)
customers shape: (250, 12)
shipments shape: (5000, 9)
order_items shape: (15022, 6)


,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,promo_code,order_subtotal,shipping_fee,tax_amount,order_total,risk_score,is_fraud
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,None,662.9500,15.4400,46.3000,724.6900,38.3000,0
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,SAVE10,862.9200,14.7400,66.6100,944.2700,94.9000,0
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,None,796.0900,14.0400,40.7200,850.8500,53.8000,1
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,WELCOME,137.6000,6.9900,11.8800,156.4700,4.2000,0
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,None,17.0700,6.9900,1.4000,25.4600,4.9000,0


In [2]:
# Target distribution and data quality checks
n_orders = len(orders_df)
fraud_counts = orders_df["is_fraud"].value_counts().sort_index()
fraud_rate = orders_df["is_fraud"].mean()

print("Total orders:", n_orders)
print("\nFraud class counts:")
print(fraud_counts)
print(f"\nFraud prevalence: {fraud_rate:.2%}")

missing_pct = (orders_df.isna().mean() * 100).sort_values(ascending=False)
print("\nMissingness (% of rows):")
print(missing_pct[missing_pct > 0])

print("\nDtypes:")
print(orders_df.dtypes)

Total orders: 5000

Fraud class counts:
is_fraud
0    4682
1     318
Name: count, dtype: int64

Fraud prevalence: 6.36%

Missingness (% of rows):
promo_code   74.7800
dtype: float64

Dtypes:
order_id            int64
customer_id         int64
order_datetime     object
billing_zip        object
shipping_zip       object
shipping_state     object
payment_method     object
device_type        object
ip_country         object
promo_used          int64
promo_code         object
order_subtotal    float64
shipping_fee      float64
tax_amount        float64
order_total       float64
risk_score        float64
is_fraud            int64
dtype: object


In [3]:
# Feature-level exploration inside orders
categorical_cols = [
    "payment_method", "device_type", "ip_country", "shipping_state", "promo_used"
]

for col in categorical_cols:
    rates = (
        orders_df.groupby(col)["is_fraud"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate", "count": "n_orders"})
        .sort_values("fraud_rate", ascending=False)
    )
    print(f"\nFraud rate by {col}:")
    print(rates)

numeric_cols = ["order_subtotal", "shipping_fee", "tax_amount", "order_total", "risk_score"]
print("\nNumeric feature means by class:")
print(orders_df.groupby("is_fraud")[numeric_cols].mean())

print("\nNumeric feature medians by class:")
print(orders_df.groupby("is_fraud")[numeric_cols].median())


Fraud rate by payment_method:
                fraud_rate  n_orders
payment_method                      
crypto              0.1031        97
card                0.0675      3128
bank                0.0593       725
paypal              0.0514      1050

Fraud rate by device_type:
             fraud_rate  n_orders
device_type                      
mobile           0.0680      2734
tablet           0.0659       364
desktop          0.0568      1902

Fraud rate by ip_country:
            fraud_rate  n_orders
ip_country                      
IN              0.0947        95
GB              0.0865       104
BR              0.0732        41
CA              0.0688       218
NG              0.0652        46
US              0.0621      4496

Fraud rate by shipping_state:
                fraud_rate  n_orders
shipping_state                      
CO                  0.0917      1702
TX                  0.0829       350
OR                  0.0755       106
VA                  0.0638        47
WA   

In [4]:
# Relationship discovery across tables
item_agg_df = pd.read_sql_query(
    """
    SELECT
        oi.order_id,
        COUNT(*) AS item_count,
        SUM(oi.quantity) AS total_qty,
        AVG(oi.unit_price) AS avg_unit_price,
        SUM(oi.line_total) AS line_total_sum
    FROM order_items oi
    GROUP BY oi.order_id
    """,
    conn,
)

analysis_df = (
    orders_df
    .merge(item_agg_df, on="order_id", how="left")
    .merge(
        shipments_df[[
            "order_id", "carrier", "shipping_method", "distance_band",
            "promised_days", "actual_days", "late_delivery"
        ]],
        on="order_id",
        how="left",
    )
)

corr_cols = [
    "is_fraud", "risk_score", "order_total", "item_count", "total_qty",
    "avg_unit_price", "promised_days", "actual_days", "late_delivery"
]

print("Correlations with is_fraud:")
print(
    analysis_df[corr_cols]
    .corr(numeric_only=True)["is_fraud"]
    .sort_values(ascending=False)
)

for col in ["shipping_method", "carrier", "distance_band", "late_delivery"]:
    rates = (
        analysis_df.groupby(col)["is_fraud"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate", "count": "n_orders"})
        .sort_values("fraud_rate", ascending=False)
    )
    print(f"\nFraud rate by {col}:")
    print(rates)

Correlations with is_fraud:
is_fraud          1.0000
actual_days       0.3205
risk_score        0.2701
late_delivery     0.2129
order_total       0.2062
total_qty         0.1412
item_count        0.1229
avg_unit_price    0.1026
promised_days    -0.0010
Name: is_fraud, dtype: float64

Fraud rate by shipping_method:
                 fraud_rate  n_orders
shipping_method                      
overnight            0.0792       303
standard             0.0638      3618
expedited            0.0584      1079

Fraud rate by carrier:
         fraud_rate  n_orders
carrier                      
USPS         0.0672       997
FedEx        0.0639      1705
UPS          0.0618      2298

Fraud rate by distance_band:
               fraud_rate  n_orders
distance_band                      
national           0.0727       990
local              0.0688      1716
regional           0.0558      2294

Fraud rate by late_delivery:
               fraud_rate  n_orders
late_delivery                      
1       

## Key Data Understanding Findings (from current dataset)
- Fraud prevalence is about **6.36%** (class imbalance exists).
- `risk_score` and `order_total` are materially higher for fraudulent orders.
- Fraud rates vary by `payment_method`, `ip_country`, and `shipping_method`.
- Aggregated basket features (`item_count`, `total_qty`, `avg_unit_price`) have signal.
- Shipment outcomes such as `late_delivery` show a strong relationship with fraud in this dataset.

## Next CRISP-DM phases (to build full production pipeline)
### 3) Data Preparation
- Define prediction-time-safe feature set.
- Build preprocessing pipeline (imputation, encoding, scaling where needed).
- Create train/validation/test split strategy (consider time-aware split using `order_datetime`).

### 4) Modeling
- Baseline: Logistic Regression.
- Candidate models: Random Forest, Gradient Boosting / XGBoost-like model.
- Handle imbalance using class weights and/or resampling.

### 5) Evaluation
- Optimize threshold using business cost matrix (false negatives vs false positives).
- Report PR-AUC, Recall, Precision, F1, ROC-AUC, and confusion matrix.
- Validate model calibration and stability over time windows.

### 6) Deployment
- Save model + preprocessing artifacts as one reproducible pipeline.
- Define scoring interface (batch or API).
- Add monitoring: data drift, concept drift, fraud-rate shift, and threshold re-tuning cadence.

---

When done with exploration, close DB connection in a final cell:
```python
conn.close()
```